In [2]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
import glob
import os
from collections import Counter
import zipfile
import shutil
import warnings
warnings.filterwarnings('ignore')

try:
    import cupy as cp
    GPU_AVAILABLE = True
    print("✓ GPU DETECTED")
except:
    GPU_AVAILABLE = False
    print("○ Using CPU")

# ============================================================================
# COLORS
# ============================================================================

REGION_COLORS = ['#d62728', '#ff7e00', '#FFD700', '#2ca02c', '#1f77ff']
REGION_LABELS_SHORT = ['R1: Red', 'R2: Orange', 'R3: Gold', 'R4: Green', 'R5: Blue']
REGION_BOUNDARIES = [(0, 23), (23, 30), (30, 46), (46, 53), (53, 60)]

CONTACT_COLORS = {
    'Very Strong': '#990000',
    'Strong': '#FF1A1A',
    'Moderate': '#FF6666',
    'Weak': '#FFB3B3',
    'No Contact': '#FFFFFF'
}

# ============================================================================
# DNA FOLDING FUNCTIONS
# ============================================================================

def calculate_radius_of_gyration(positions):
    try:
        center = np.mean(positions, axis=0)
        rg = np.sqrt(np.mean(np.sum((positions - center)**2, axis=1)))
        return rg if not np.isnan(rg) else 15.0
    except:
        return 15.0

def calculate_end_to_end_distance(positions):
    try:
        dist = np.linalg.norm(positions[-1] - positions[0])
        return dist if not np.isnan(dist) else 80.0
    except:
        return 80.0

def calculate_dna_curvature(positions):
    try:
        vectors = positions[1:] - positions[:-1]
        angles = []
        for i in range(len(vectors)-1):
            v1, v2 = vectors[i], vectors[i+1]
            norm_product = np.linalg.norm(v1) * np.linalg.norm(v2)
            if norm_product > 1e-10:
                cos_angle = np.clip(np.dot(v1, v2) / norm_product, -1, 1)
                angle = np.arccos(cos_angle)
                if not np.isnan(angle):
                    angles.append(angle)
        return np.mean(angles) if len(angles) > 0 else 0.15
    except:
        return 0.15

def simulate_dna_positions(contact_matrix):
    n_dna = contact_matrix.shape[1]
    positions = []
    max_contact = max(np.max(contact_matrix), 1.0)
    
    for i in range(n_dna):
        contact_sum = np.sum(contact_matrix[:, i])
        radius = 20.0 - (contact_sum / max_contact * 8.0)
        angle = (i / n_dna) * 2 * np.pi
        x = radius * np.cos(angle)
        y = radius * np.sin(angle)
        z = i * 0.5
        positions.append([x, y, z])
    
    return np.array(positions)

# ============================================================================
# HELPERS
# ============================================================================

def classify_contact_frequency(total_contact, max_contact):
    if max_contact > 0:
        normalized = total_contact / max_contact
    else:
        normalized = 0
    
    if normalized >= 0.8:
        return 'Very Strong'
    elif normalized >= 0.6:
        return 'Strong'
    elif normalized >= 0.4:
        return 'Moderate'
    elif normalized >= 0.2:
        return 'Weak'
    else:
        return 'No Contact'

def gpu_kmeans_inertia(features, centers, labels):
    if GPU_AVAILABLE:
        try:
            features_gpu = cp.asarray(features)
            centers_gpu = cp.asarray(centers)
            distances = cp.linalg.norm(features_gpu[:, None, :] - centers_gpu[None, :, :], axis=2)
            min_distances = cp.min(distances, axis=1)
            return float(cp.sum(min_distances ** 2))
        except:
            pass
    distances = np.linalg.norm(features[:, None, :] - centers[None, :, :], axis=2)
    min_distances = np.min(distances, axis=1)
    return np.sum(min_distances ** 2)

# ============================================================================
# FEATURE EXTRACTION
# ============================================================================

def extract_regional_features_with_folding(contact_matrices, time_labels):
    print("\nExtracting Regional Features (with DNA folding)...")
    print(f"Processing {len(contact_matrices)} frames")
    print(f"Extracting 5 regions per frame → {len(contact_matrices) * 5} total data points")
    
    features_list = []
    region_labels = []
    contact_strengths = []
    frame_ids = []
    time_ids = []
    folding_metrics = []
    
    max_total_contact = max([np.sum(m) for m in contact_matrices] + [1.0])
    
    for frame_idx, matrix in enumerate(contact_matrices):
        positions = simulate_dna_positions(matrix)
        rg = calculate_radius_of_gyration(positions)
        end_to_end = calculate_end_to_end_distance(positions)
        curvature = calculate_dna_curvature(positions)
        
        for region_id, (start, end) in enumerate(REGION_BOUNDARIES):
            region_matrix = matrix[:, start:end]
            region_contact = np.sum(region_matrix)
            region_id_norm = region_id / 4.0
            frame_id_norm = frame_idx / max(len(contact_matrices), 1)
            strength = classify_contact_frequency(region_contact, max_total_contact)
            
            features_list.append([
                float(region_contact),
                float(region_id_norm),
                float(frame_id_norm),
                float(rg),
                float(end_to_end),
                float(curvature)
            ])
            
            region_labels.append(region_id)
            contact_strengths.append(strength)
            frame_ids.append(frame_idx)
            time_ids.append(time_labels[frame_idx])
            folding_metrics.append({'rg': rg, 'e2e': end_to_end, 'curv': curvature})
    
    features_array = np.array(features_list, dtype=np.float64)
    
    if np.any(np.isnan(features_array)):
        print("  Fixing NaN values...")
        imputer = SimpleImputer(strategy='mean')
        features_array = imputer.fit_transform(features_array)
    
    print(f"✓ Extracted {len(features_list)} regional data points")
    
    return (features_array, region_labels, contact_strengths, frame_ids, 
            time_ids, folding_metrics)

# ============================================================================
# MAIN FUNCTION
# ============================================================================

def gpu_accelerated_regional_clustering(npy_dir, output_dir="/kaggle/working/clustering_results"):
    
    print("\n" + "="*70)
    print("GPU-ACCELERATED REGIONAL K-MEANS CLUSTERING + DNA FOLDING")
    print("="*70)
    
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    
    plots_dir = os.path.join(output_dir, "plots")
    data_dir = os.path.join(output_dir, "data")
    
    for d in [plots_dir, data_dir]:
        os.makedirs(d, exist_ok=True)
    
    # ★★★ STEP 1: LOAD NPZ FILES WITH NaN/Inf HANDLING ★★★
    npy_files = sorted(glob.glob(os.path.join(npy_dir, "*.npz")))
    
    if len(npy_files) == 0:
        print(f"ERROR: No NPZ files found in {npy_dir}")
        return None
    
    print(f"\nStep 1: Loading {len(npy_files)} NPZ files")
    
    contact_matrices = []
    time_labels = []
    valid_count = 0
    nan_count = 0
    zero_count = 0
    error_count = 0
    
    for i, file_path in enumerate(npy_files):
        try:
            data = np.load(file_path)
            
            if 'matrix' not in data:
                error_count += 1
                continue
            
            matrix = data['matrix']
            
            # ★★★ FIX: Handle NaN/Inf values ★★★
            has_nan = np.any(np.isnan(matrix))
            has_inf = np.any(np.isinf(matrix))
            
            if has_nan or has_inf:
                nan_count += 1
                matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)
            
            if np.all(matrix == 0):
                zero_count += 1
            
            # ★★★ FIX: Extract time safely ★★★
            if 't_start_ns' in data:
                try:
                    time_val = data['t_start_ns']
                    if isinstance(time_val, np.ndarray):
                        time_val = time_val.item()
                    time = int(time_val) if not np.isnan(time_val) else len(contact_matrices)
                except:
                    time = len(contact_matrices)
            else:
                time = len(contact_matrices)
            
            contact_matrices.append(matrix.astype(np.float32))
            time_labels.append(time)
            valid_count += 1
            
        except Exception as e:
            error_count += 1
            if i < 3:  # Only print first 3 errors
                print(f"  Warning: File {i} error: {e}")
            continue
    
    # ★★★ SUMMARY ★★★
    print(f"\n📊 Loading Summary:")
    print(f"  ✓ Total files: {len(npy_files)}")
    print(f"  ✓ Valid matrices: {valid_count}")
    print(f"  ✓ Had NaN/Inf (fixed): {nan_count}")
    print(f"  ✓ Zero matrices: {zero_count}")
    print(f"  ✓ Non-zero matrices: {valid_count - zero_count}")
    print(f"  ✓ Errors/skipped: {error_count}")
    
    if valid_count == 0:
        print("\n❌ ERROR: No valid matrices loaded!")
        return None
    
    if zero_count == valid_count:
        print("\n⚠️  WARNING: All matrices are zeros - clustering will show 'no contact' states")
    
    # Extract features
    print(f"\nStep 2: Feature Engineering")
    (features_array, region_labels, contact_strengths, frame_ids, 
     time_ids, folding_metrics) = extract_regional_features_with_folding(
        contact_matrices, time_labels
    )
    
    scaler = StandardScaler()
    features_normalized = scaler.fit_transform(features_array)
    features_normalized = np.nan_to_num(features_normalized, nan=0.0)
    
    # Optimal K
    print(f"\nStep 3: Finding Optimal K")
    
    max_k = min(10, max(2, len(features_array) // 50))
    k_range = range(2, max_k + 1)
    silhouette_scores = []
    inertias = []
    
    for k in k_range:
        if k > len(features_array):
            break
            
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=300)
        labels = kmeans.fit_predict(features_normalized)
        centers = kmeans.cluster_centers_
        
        try:
            sil = silhouette_score(features_normalized, labels)
        except:
            sil = 0.0
            
        inertia = gpu_kmeans_inertia(features_normalized, centers, labels)
        silhouette_scores.append(sil)
        inertias.append(inertia)
        print(f"  K={k}: Silhouette={sil:.3f}")
    
    if len(silhouette_scores) == 0:
        print("ERROR: Could not compute clustering")
        return None
        
    optimal_k = k_range[np.argmax(silhouette_scores)]
    print(f"\n✓ Optimal K = {optimal_k}")
    
    # Plot optimization
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].plot(list(k_range)[:len(silhouette_scores)], silhouette_scores, 'bo-', linewidth=2, markersize=8)
    axes[0].axvline(optimal_k, color='red', linestyle='--', alpha=0.7, linewidth=2, label=f'Optimal K={optimal_k}')
    axes[0].set_xlabel('Number of Clusters (K)', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Silhouette Score', fontsize=13, fontweight='bold')
    axes[0].set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(list(k_range)[:len(inertias)], inertias, 'go-', linewidth=2, markersize=8)
    axes[1].axvline(optimal_k, color='red', linestyle='--', alpha=0.7, linewidth=2, label=f'Optimal K={optimal_k}')
    axes[1].set_xlabel('Number of Clusters (K)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Inertia', fontsize=13, fontweight='bold')
    axes[1].set_title('Elbow Method', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    axes[2].bar(list(k_range)[:len(silhouette_scores)], silhouette_scores, color='skyblue', edgecolor='black', linewidth=1.5)
    axes[2].axvline(optimal_k - 0.5, color='red', linestyle='--', linewidth=3, label=f'Best K={optimal_k}')
    axes[2].set_xlabel('Number of Clusters (K)', fontsize=13, fontweight='bold')
    axes[2].set_ylabel('Silhouette Score', fontsize=13, fontweight='bold')
    axes[2].set_title('Best K Selection', fontsize=14, fontweight='bold')
    axes[2].legend(fontsize=11)
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '01_cluster_optimization.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # Final clustering
    print(f"\nStep 4: Final Clustering (K={optimal_k})")
    
    final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=50, max_iter=500)
    cluster_labels = final_kmeans.fit_predict(features_normalized)
    cluster_centers = final_kmeans.cluster_centers_
    
    final_silhouette = silhouette_score(features_normalized, cluster_labels)
    print(f"✓ Silhouette Score: {final_silhouette:.3f}")
    
    # Folding stats
    folding_stats = {}
    for c in range(optimal_k):
        mask = cluster_labels == c
        c_folding = [folding_metrics[i] for i in range(len(mask)) if mask[i]]
        folding_stats[c] = {
            'rg': np.mean([f['rg'] for f in c_folding]),
            'e2e': np.mean([f['e2e'] for f in c_folding]),
            'curv': np.mean([f['curv'] for f in c_folding]),
            'size': np.sum(mask)
        }
    
    rg_all = np.array([f['rg'] for f in folding_metrics])
    e2e_all = np.array([f['e2e'] for f in folding_metrics])
    contact_all = features_array[:, 0]
    
    print(f"\nStep 5: Creating Visualizations")
    
    # Main clustering figure
    fig = plt.figure(figsize=(22, 15))
    
    # Plot 1: Regional DNA
    ax1 = fig.add_subplot(231, projection='3d')
    for region_id in range(5):
        mask = np.array(region_labels) == region_id
        if not np.any(mask):
            continue
        points = features_normalized[mask]
        color = REGION_COLORS[region_id]
        ax1.scatter(points[:, 0], points[:, 1], points[:, 2],
                   c=color, s=50, alpha=0.7,
                   edgecolors='black', linewidth=0.5)
    
    ax1.set_xlabel('Regional Contact', fontsize=11, color='#d62728', fontweight='bold')
    ax1.set_ylabel('Region ID', fontsize=11, color='#ff7e00', fontweight='bold')
    ax1.set_zlabel('Frame ID', fontsize=11, color='#FFD700', fontweight='bold')
    ax1.set_title('Regional DNA Classification', fontsize=12, fontweight='bold', pad=20)
    ax1.grid(True, alpha=0.3)
    
    legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor=color,
                             markersize=10, markeredgecolor='black', markeredgewidth=1.5,
                             label=label)
                      for color, label in zip(REGION_COLORS, REGION_LABELS_SHORT)]
    ax1.legend(handles=legend_elements, loc='upper left', fontsize=9,
              title='DNA Regions', title_fontsize=10, framealpha=0.95, edgecolor='black')
    
    # Plot 2: K-means Clustering
    ax2 = fig.add_subplot(232, projection='3d')
    cluster_colors_map = plt.cm.Set1(np.linspace(0, 1, optimal_k))
    
    for cluster_id in range(optimal_k):
        mask = cluster_labels == cluster_id
        points = features_normalized[mask]
        ax2.scatter(points[:, 0], points[:, 1], points[:, 2],
                   c=[cluster_colors_map[cluster_id]], s=50, alpha=0.7,
                   edgecolors='black', linewidth=0.5,
                   label=f'C{cluster_id + 1} ({np.sum(mask)} pts)')
    
    ax2.scatter(cluster_centers[:, 0], cluster_centers[:, 1], cluster_centers[:, 2],
               c='black', s=300, marker='X',
               edgecolors='white', linewidth=2,
               label='Centroids', zorder=10)
    
    ax2.set_xlabel('Regional Contact', fontsize=11, color='#d62728', fontweight='bold')
    ax2.set_ylabel('Region ID', fontsize=11, color='#ff7e00', fontweight='bold')
    ax2.set_zlabel('Frame ID', fontsize=11, color='#FFD700', fontweight='bold')
    ax2.set_title(f'K-means Clustering (K={optimal_k})\nSilhouette={final_silhouette:.3f}',
                 fontsize=12, fontweight='bold', pad=20)
    ax2.legend(loc='upper left', fontsize=9, framealpha=0.95, edgecolor='black')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Contact Strength
    ax3 = fig.add_subplot(233, projection='3d')
    
    for i in range(len(features_normalized)):
        strength = contact_strengths[i]
        color = CONTACT_COLORS[strength]
        ax3.scatter(features_normalized[i, 0], features_normalized[i, 1], features_normalized[i, 2],
                   c=color, s=50, alpha=0.8, edgecolors='black', linewidth=0.5)
    
    for cluster_id in range(optimal_k):
        ax3.scatter(cluster_centers[cluster_id, 0], cluster_centers[cluster_id, 1], cluster_centers[cluster_id, 2],
                   c='black', s=300, marker='X', edgecolors='white', linewidth=2)
    
    ax3.set_xlabel('Regional Contact', fontsize=11, color='#d62728', fontweight='bold')
    ax3.set_ylabel('Region ID', fontsize=11, color='#ff7e00', fontweight='bold')
    ax3.set_zlabel('Frame ID', fontsize=11, color='#FFD700', fontweight='bold')
    ax3.set_title('Contact Strength Classification',
                 fontsize=12, fontweight='bold', pad=20)
    ax3.grid(True, alpha=0.3)
    
    strength_legend = [Line2D([0], [0], marker='o', color='w', markerfacecolor=color,
                             markersize=8, markeredgecolor='black', markeredgewidth=1.5,
                             label=f'{strength_name} ({sum(1 for s in contact_strengths if s == strength_name)})')
                      for strength_name, color in CONTACT_COLORS.items()]
    ax3.legend(handles=strength_legend, loc='upper left', fontsize=9,
              title='Contact Strength', title_fontsize=10, framealpha=0.95, edgecolor='black')
    
    # Plot 4: Time Evolution
    ax4 = fig.add_subplot(234)
    
    for i in range(len(time_ids)):
        strength = contact_strengths[i]
        color = CONTACT_COLORS[strength]
        cluster = cluster_labels[i]
        ax4.scatter(time_ids[i], cluster, c=color, s=80, alpha=0.8,
                   edgecolors='black', linewidth=0.8, zorder=3)
    
    ax4.set_xlabel('Time (ns)', fontsize=13, fontweight='bold')
    ax4.set_ylabel('Cluster Assignment', fontsize=13, fontweight='bold')
    ax4.set_title('Cluster Evolution Over Time',
                 fontsize=12, fontweight='bold')
    ax4.set_yticks(range(optimal_k))
    ax4.set_yticklabels([f'C{i+1}' for i in range(optimal_k)])
    
    for idx, tick in enumerate(ax4.get_yticklabels()):
        tick.set_color(REGION_COLORS[idx % 5])
        tick.set_fontweight('bold')
        tick.set_fontsize(12)
    
    ax4.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
    ax4.set_axisbelow(True)
    
    # Plot 5: Regional Composition
    ax5 = fig.add_subplot(235)
    
    cluster_regional_data = []
    for cluster_id in range(optimal_k):
        mask = cluster_labels == cluster_id
        cluster_regions = [region_labels[i] for i in range(len(mask)) if mask[i]]
        region_counts = Counter(cluster_regions)
        cluster_regional_data.append(region_counts)
    
    x_pos = np.arange(optimal_k)
    bottom = np.zeros(optimal_k)
    
    for region_id in range(5):
        heights = [cluster_regional_data[c].get(region_id, 0) for c in range(optimal_k)]
        ax5.bar(x_pos, heights, bottom=bottom,
               color=REGION_COLORS[region_id],
               edgecolor='black', linewidth=0.8,
               label=REGION_LABELS_SHORT[region_id])
        bottom += heights
    
    ax5.set_xlabel('Cluster ID', fontsize=13, fontweight='bold')
    ax5.set_ylabel('Number of Data Points', fontsize=13, fontweight='bold')
    ax5.set_title('Regional Composition per Cluster',
                 fontsize=12, fontweight='bold')
    ax5.set_xticks(x_pos)
    ax5.set_xticklabels([f'C{i+1}' for i in range(optimal_k)])
    
    for idx, tick in enumerate(ax5.get_xticklabels()):
        tick.set_color(REGION_COLORS[idx % 5])
        tick.set_fontweight('bold')
    
    ax5.legend(loc='upper right', fontsize=10, title='DNA Regions', title_fontsize=10,
              framealpha=0.95, edgecolor='black')
    ax5.grid(True, alpha=0.3, axis='y')
    
    # Plot 6: Regional Distribution
    ax6 = fig.add_subplot(236)
    
    region_counts = Counter(region_labels)
    region_values = [region_counts[i] for i in range(5)]
    
    wedges, texts, autotexts = ax6.pie(region_values,
                                       labels=REGION_LABELS_SHORT,
                                       colors=REGION_COLORS,
                                       autopct='%1.1f%%',
                                       startangle=90,
                                       textprops={'fontsize': 10, 'fontweight': 'bold'})
    
    for autotext in autotexts:
        autotext.set_color('black')
        autotext.set_fontsize(11)
    
    ax6.set_title('Overall Regional Distribution',
                 fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, '02_regional_clustering.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    
    print("  Saved: 02_regional_clustering.png")
    
    # Save data
    np.savez_compressed(
        os.path.join(data_dir, 'clustering_results.npz'),
        cluster_labels=cluster_labels,
        features=features_array,
        rg=rg_all,
        optimal_k=optimal_k
    )
    
    # ZIP
    zip_path = "/kaggle/working/regional_clustering_results.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(output_dir):
            for file in files:
                zipf.write(os.path.join(root, file),
                          os.path.relpath(os.path.join(root, file), output_dir))
    
    print(f"\n✅ COMPLETE!")
    print(f"   ZIP: {zip_path}")
    return zip_path

# ============================================================================
# RUN
# ============================================================================

zip_file = gpu_accelerated_regional_clustering(
    npy_dir="/kaggle/input/npy-system-2/npy"
)


✓ GPU DETECTED

GPU-ACCELERATED REGIONAL K-MEANS CLUSTERING + DNA FOLDING

Step 1: Loading 202 NPZ files

📊 Loading Summary:
  ✓ Total files: 202
  ✓ Valid matrices: 202
  ✓ Had NaN/Inf (fixed): 0
  ✓ Zero matrices: 27
  ✓ Non-zero matrices: 175
  ✓ Errors/skipped: 0

Step 2: Feature Engineering

Extracting Regional Features (with DNA folding)...
Processing 202 frames
Extracting 5 regions per frame → 1010 total data points
✓ Extracted 1010 regional data points

Step 3: Finding Optimal K
  K=2: Silhouette=0.444
  K=3: Silhouette=0.480
  K=4: Silhouette=0.486
  K=5: Silhouette=0.458
  K=6: Silhouette=0.478
  K=7: Silhouette=0.490
  K=8: Silhouette=0.454
  K=9: Silhouette=0.432
  K=10: Silhouette=0.433

✓ Optimal K = 7

Step 4: Final Clustering (K=7)
✓ Silhouette Score: 0.490

Step 5: Creating Visualizations
  Saved: 02_regional_clustering.png

✅ COMPLETE!
   ZIP: /kaggle/working/regional_clustering_results.zip
